# Silver layer

# Preprocessing for text cleaning is as follows:
- Normalize line endings
- Replace tabs and non-breaking spaces
- Remove zero-width spaces if present
- Fix hyphenated line breaks
- Remove trailing spaces on lines
- Remove repeated spaces inside lines
- Normalize spaces around line breaks
- Merge broken PDF lines where useful for reading
- Remove common PDF page markers
- Remove reference-like lines
- Remove duplicate lines where useful for reading
- Reduce too many blank lines

# We create two cleaned text versions:
- cleaned_text: reading-friendly text for summarization / general downstream use
- nlp_text: structure-preserving text for NLP / entity extraction / front-matter logic

Input: bronze JSON
Output: silver JSON with cleaned text fields

In [54]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

In [55]:
BRONZE_FOLDER = Path("../../Data/bronze")
SILVER_FOLDER = Path("../../Data/silver")
REPORTS_FOLDER = SILVER_FOLDER / "reports"

SILVER_FOLDER.mkdir(parents=True, exist_ok=True)
REPORTS_FOLDER.mkdir(parents=True, exist_ok=True)

# Helpers

In [56]:
def safe_string(text: Any) -> str:
    if pd.isna(text):
        return ""
    return str(text)


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, payload: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

# Clean incoming text data

In [57]:
def normalize_line_endings(text: Any) -> str:
    text = safe_string(text)
    return text.replace("\r\n", "\n").replace("\r", "\n")


def replace_special_spaces(text: Any) -> str:
    text = safe_string(text)
    return text.replace("\t", " ").replace("\xa0", " ")


def remove_zero_width_characters(text: Any) -> str:
    text = safe_string(text)
    for char in ["\u200B", "\u200C", "\u200D", "\uFEFF"]:
        text = text.replace(char, "")
    return text


def fix_hyphenated_linebreaks(text: Any) -> str:
    text = safe_string(text)
    return re.sub(r"(\w)-\n(\w)", r"\1\2", text)


def remove_pdf_page_markers(text: Any) -> str:
    text = safe_string(text)

    patterns = [
        r"^\s*\d+\s*$",                    # page numbers only
        r"^\s*Page\s+\d+\s*$",
        r"^\s*Page\s+\d+\s+of\s+\d+\s*$",
        r"^\s*P\s*a\s*g\s*e\s+\d+\s*$",
    ]

    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE | re.MULTILINE)

    return text


def remove_urls_and_emails(text: Any) -> str:
    text = safe_string(text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", "", text)
    return text


def remove_trailing_spaces(text: Any) -> str:
    text = safe_string(text)
    return re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)


def remove_repeated_spaces(text: Any) -> str:
    text = safe_string(text)
    return re.sub(r" {2,}", " ", text)


def normalize_linebreak_spacing(text: Any) -> str:
    text = safe_string(text)
    return re.sub(r" *\n *", "\n", text)


def normalize_linebreak_spacing_soft(text: Any) -> str:
    text = safe_string(text)
    return re.sub(r"[ \t]*\n[ \t]*", "\n", text)


def merge_broken_lines(text: Any) -> str:
    text = safe_string(text)
    lines = text.split("\n")
    merged = []

    for line in lines:
        line = line.strip()

        if not line:
            merged.append("")
            continue

        if not merged:
            merged.append(line)
            continue

        prev = merged[-1]

        # preserve likely headings
        if len(prev.split()) <= 8:
            merged.append(line)
            continue

        # preserve short standalone lines
        if len(line.split()) <= 4 and line[:1].isupper():
            merged.append(line)
            continue

        # preserve bullet / numbered items
        if re.match(r"^(\d+[\.\)]\s+|[-•]\s+)", line):
            merged.append(line)
            continue

        # merge likely wrapped paragraph lines
        if not prev.endswith((".", "!", "?", ":", ";")) and line[:1].islower():
            merged[-1] = prev + " " + line
        else:
            merged.append(line)

    return "\n".join(merged)


def remove_reference_like_lines(text: Any) -> str:
    text = safe_string(text)
    cleaned_lines = []

    for line in text.split("\n"):
        stripped = line.strip()

        if not stripped:
            cleaned_lines.append("")
            continue

        if re.match(r"^[A-Z][a-zA-Z\-]+,\s+[A-Z]\.", stripped):
            continue
        if re.search(r"\(\d{4}\)", stripped) and len(stripped.split()) < 12:
            continue
        if stripped.lower().startswith("references"):
            continue
        if stripped.lower().startswith("bibliography"):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def remove_duplicate_lines(text: Any) -> str:
    text = safe_string(text)
    seen = set()
    result = []

    for line in text.split("\n"):
        key = re.sub(r"\s+", " ", line.strip().lower())

        if key and key in seen:
            continue
        if key:
            seen.add(key)

        result.append(line)

    return "\n".join(result)


def remove_many_blank_lines(text: Any) -> str:
    text = safe_string(text)
    return re.sub(r"\n{3,}", "\n\n", text)


Run all functions on text

In [58]:
def clean_text_for_reading(text: Any) -> str:
    text = normalize_line_endings(text)
    text = replace_special_spaces(text)
    text = remove_zero_width_characters(text)
    text = fix_hyphenated_linebreaks(text)
    text = remove_pdf_page_markers(text)
    text = remove_urls_and_emails(text)
    text = remove_trailing_spaces(text)
    text = remove_repeated_spaces(text)
    text = normalize_linebreak_spacing(text)
    text = merge_broken_lines(text)
    text = remove_reference_like_lines(text)
    text = remove_duplicate_lines(text)
    text = remove_many_blank_lines(text)
    return text.strip()


def clean_text_for_nlp(text: Any) -> str:
    text = normalize_line_endings(text)
    text = replace_special_spaces(text)
    text = remove_zero_width_characters(text)
    text = fix_hyphenated_linebreaks(text)
    text = remove_pdf_page_markers(text)
    text = remove_urls_and_emails(text)
    text = remove_trailing_spaces(text)
    text = remove_repeated_spaces(text)
    text = normalize_linebreak_spacing_soft(text)
    text = remove_reference_like_lines(text)
    text = remove_many_blank_lines(text)
    return text.strip()

Run sanity checks

In [59]:
def build_quality_metrics(raw_text: str, cleaned_text: str, nlp_text: str) -> dict[str, Any]:
    raw_length = len(raw_text)
    cleaned_length = len(cleaned_text)
    nlp_length = len(nlp_text)

    cleaned_ratio = cleaned_length / raw_length if raw_length else 0
    nlp_ratio = nlp_length / raw_length if raw_length else 0

    flags = []
    if raw_length == 0:
        flags.append("empty_raw_text")
    if cleaned_length < 200:
        flags.append("very_short_cleaned_text")
    if cleaned_ratio < 0.30 and raw_length > 500:
        flags.append("aggressive_cleaning_possible")
    if nlp_length < 200:
        flags.append("very_short_nlp_text")

    return {
        "raw_length": raw_length,
        "cleaned_length": cleaned_length,
        "nlp_length": nlp_length,
        "cleaned_ratio": round(cleaned_ratio, 4),
        "nlp_ratio": round(nlp_ratio, 4),
        "quality_flags": flags,
    }


def sanity_preview(raw_text: str, cleaned_text: str, nlp_text: str, n_chars: int = 700) -> None:
    print("Original Text:\n")
    print(raw_text[:n_chars])

    print("\n" + "-" * 80 + "\n")
    print("Cleaned Text for Reading:\n")
    print(cleaned_text[:n_chars])

    print("\n" + "-" * 80 + "\n")
    print("Cleaned Text for NLP:\n")
    print(nlp_text[:n_chars])


In [60]:
def process_bronze_record(record: dict[str, Any]) -> dict[str, Any]:
    raw_text = safe_string(record.get("raw_text"))

    cleaned_text = clean_text_for_reading(raw_text)
    nlp_text = clean_text_for_nlp(raw_text)
    metrics = build_quality_metrics(raw_text, cleaned_text, nlp_text)

    return {
        "document_id": record.get("document_id"),
        "source_file_name": record.get("source_file_name"),
        "source_file_path": record.get("source_file_path"),
        "raw_text": raw_text,
        "cleaned_text": cleaned_text,
        "nlp_text": nlp_text,
        "pdf_metadata": record.get("pdf_metadata", {}),
        "page_count": record.get("page_count"),
        "page_extraction_log": record.get("page_extraction_log", []),
        "bronze_extraction_status": record.get("extraction_status"),
        "used_ocr_fallback": record.get("used_ocr_fallback", False),
        "bronze_processed_at_utc": record.get("processed_at_utc"),
        "silver_processed_at_utc": datetime.now(timezone.utc).isoformat(),
        **metrics,
    }


In [61]:
bronze_files = sorted(
    p for p in BRONZE_FOLDER.glob("*_bronze.json")
    if p.is_file()
)

if not bronze_files:
    raise FileNotFoundError(f"No Bronze JSON files found in {BRONZE_FOLDER}")

results = []
manifest = []

for bronze_path in bronze_files:
    payload = load_json(bronze_path)

    if isinstance(payload, list):
        if not payload:
            print(f"Skipping empty Bronze file: {bronze_path.name}")
            continue
        bronze_record = payload[0]
    elif isinstance(payload, dict):
        bronze_record = payload
    else:
        print(f"Skipping unsupported JSON format: {bronze_path.name}")
        continue

    required_columns = ["document_id", "raw_text"]
    missing_columns = [col for col in required_columns if col not in bronze_record]
    if missing_columns:
        raise ValueError(f"{bronze_path.name} missing required fields: {missing_columns}")

    silver_record = process_bronze_record(bronze_record)

    output_path = SILVER_FOLDER / f"{silver_record['document_id']}_silver.json"
    write_json(output_path, [silver_record])

    manifest.append({
        "document_id": silver_record["document_id"],
        "source_file_name": silver_record["source_file_name"],
        "source_file_path": silver_record["source_file_path"],
        "output_json_path": str(output_path.resolve()),
        "raw_length": silver_record["raw_length"],
        "cleaned_length": silver_record["cleaned_length"],
        "nlp_length": silver_record["nlp_length"],
        "cleaned_ratio": silver_record["cleaned_ratio"],
        "nlp_ratio": silver_record["nlp_ratio"],
        "quality_flags": silver_record["quality_flags"],
        "silver_processed_at_utc": silver_record["silver_processed_at_utc"],
    })

    results.append(silver_record)
    print(f"Saved Silver JSON: {output_path.name}")


Saved Silver JSON: doc_01_silver.json


In [62]:
if results:
    first = results[0]
    sanity_preview(first["raw_text"], first["cleaned_text"], first["nlp_text"])


Original Text:

TOLWEG of TOL WEG? Continueren of afschaffen van de tolheffing voor de Westerscheldetunnel  – een scenariostudie -

2 
 Tolweg of Tol weg? Continueren of afschaffen van de tolheffing voor de Westerscheldetunnel  – een scenariostudie   Definitief rapport, 10 januari 2018    Colofon  © TU Delft/  ZB 2018  Samenstelling Evert Meijers Dick van der Wouw Erik Louw Marjolein Spaans  Technische Universiteit Delft Faculteit Bouwkunde Sectie Stedelijke en Regionale Ontwikkeling Julianalaan 134 2628 BL Delft Telefoon (015) 2783450 bk.tudelft.nl   ZB| Planbureau en Bibliotheek van Zeeland Kousteensedijk 7 4331 JE Middelburg Postbus 8004  4330 EA Middelburg Telefoon (0118) 654000 www.dezb.nl info@dezb.nl

--------------------------------------------------------------------------------

Cleaned Text for Reading:

TOLWEG of TOL WEG? Continueren of afschaffen van de tolheffing voor de Westerscheldetunnel – een scenariostudie -

Tolweg of Tol weg? Continueren of afschaffen van de tolhef

In [63]:
summary_df = pd.DataFrame(results)[[
    "document_id",
    "raw_length",
    "cleaned_length",
    "nlp_length",
    "cleaned_ratio",
    "nlp_ratio",
    "quality_flags",
]]

print("\nSilver quality summary:")
print(summary_df)

print("\nDescriptive stats:")
print(summary_df[["raw_length", "cleaned_length", "nlp_length", "cleaned_ratio", "nlp_ratio"]].describe())

validation_report = {
    "total_input_files": len(bronze_files),
    "successful_documents": len(results),
    "documents_with_flags": sum(1 for r in results if r["quality_flags"]),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "documents": manifest,
}

write_json(REPORTS_FOLDER / "silver_validation_report.json", validation_report)
write_json(REPORTS_FOLDER / "silver_manifest.json", manifest)

print(f"\nSaved validation report to: {REPORTS_FOLDER / 'silver_validation_report.json'}")
print(f"Saved manifest to: {REPORTS_FOLDER / 'silver_manifest.json'}")


Silver quality summary:
  document_id  raw_length  cleaned_length  nlp_length  cleaned_ratio  \
0      doc_01      175421          165012      169508         0.9407   

   nlp_ratio quality_flags  
0     0.9663            []  

Descriptive stats:
       raw_length  cleaned_length  nlp_length  cleaned_ratio  nlp_ratio
count         1.0             1.0         1.0         1.0000     1.0000
mean     175421.0        165012.0    169508.0         0.9407     0.9663
std           NaN             NaN         NaN            NaN        NaN
min      175421.0        165012.0    169508.0         0.9407     0.9663
25%      175421.0        165012.0    169508.0         0.9407     0.9663
50%      175421.0        165012.0    169508.0         0.9407     0.9663
75%      175421.0        165012.0    169508.0         0.9407     0.9663
max      175421.0        165012.0    169508.0         0.9407     0.9663

Saved validation report to: ..\..\Data\silver\reports\silver_validation_report.json
Saved manifest to: 